# Reproducibility of the analysis

In [392]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
from sklearn.metrics import cohen_kappa_score

In [393]:
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

SEED = 42
N_BOOT = 10000

CONDITIONS = ["C0", "CL", "CO", "CD", "CS", "CT"]
CONTEXTUAL = ["CL", "CO", "CD", "CS", "CT"]

TARGET_CATEGORY = {
    "CL": "Lexical",
    "CO": "Operational",
    "CD": "Decisional",
    "CS": "Systemic",
    "CT": None,
}

## 1. Loading the data

In [394]:
# Resolve repository root regardless of where Jupyter is launched.
CWD = Path.cwd()
REPO_ROOT = CWD.parent if CWD.name == "notebooks" else CWD

PREFERRED_XLSX = REPO_ROOT / "data" / "base.xlsx"
FALLBACK_XLSX = REPO_ROOT / "results" / "freezed-results.xlsx"

if PREFERRED_XLSX.exists():
    XLSX_PATH = PREFERRED_XLSX
    print(f"Using frozen analysis dataset: {XLSX_PATH.relative_to(REPO_ROOT)}")
elif FALLBACK_XLSX.exists():
    XLSX_PATH = FALLBACK_XLSX
    print("Preferred frozen dataset not found.")
    print(f"  Expected at: {PREFERRED_XLSX}")
    print(f"  Falling back to: {XLSX_PATH.relative_to(REPO_ROOT)}")
    print("  To use the canonical location, copy the workbook to the path above.")
else:
    raise FileNotFoundError(
        "Frozen analysis dataset not found.\n"
        f"Place the workbook at:\n  {PREFERRED_XLSX}\n"
        f"or provide the fallback at:\n  {FALLBACK_XLSX}"
    )

RESULTS_DIR = REPO_ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)

Using frozen analysis dataset: data\base.xlsx


In [395]:
def read_sheet(name):
    """Read a sheet and drop fully empty rows (Excel keeps blank formatted rows)."""
    df = pd.read_excel(XLSX_PATH, sheet_name=name, engine="openpyxl")
    return df.dropna(how="all").reset_index(drop=True)


raw_sheets = pd.read_excel(XLSX_PATH, sheet_name=None, engine="openpyxl")
print("Sheets found in workbook:")
for name, df in raw_sheets.items():
    print(f"  {name}: {df.dropna(how='all').shape[0]} non-empty rows, {df.shape[1]} columns")

Sheets found in workbook:
  user_stories: 25 non-empty rows, 3 columns
  materials: 23 non-empty rows, 8 columns
  executions: 120 non-empty rows, 9 columns
  raw_info: 1039 non-empty rows, 10 columns
  judge-annotation-1: 598 non-empty rows, 13 columns
  judge-annotation-2: 598 non-empty rows, 13 columns
  Gap_normalization: 64 non-empty rows, 4 columns
  Adjudication: 598 non-empty rows, 27 columns
  Gap_convenience: 22 non-empty rows, 4 columns


## 2. Verifying the expected sheets

In [396]:
EXPECTED_SHEETS = [
    "user_stories",
    "materials",
    "executions",
    "raw_info",
    "judge-annotation-1",
    "judge-annotation-2",
    "Gap_normalization",
    "Adjudication",
    "Gap_convenience",
]

present = set(raw_sheets.keys())
missing = [s for s in EXPECTED_SHEETS if s not in present]
if missing:
    raise AssertionError(f"Missing expected sheets: {missing}")
print("All expected sheets are present.")

executions = read_sheet("executions")
raw_info = read_sheet("raw_info")
judge1 = read_sheet("judge-annotation-1")
judge2 = read_sheet("judge-annotation-2")
gap_norm = read_sheet("Gap_normalization")
adj = read_sheet("Adjudication")

All expected sheets are present.


## 3. extraction and cleaning

- `executions`: total registered runs and valid runs (`Status == "Valida"`).
- `raw_info`: rows extracted, duplicates removed, and questions kept for annotation.
  - duplicates removed = rows where `Duplicate_In_Run` starts with `"Duplicate of"`.
  - questions kept = rows where `Duplicate_In_Run` is `"Primary"` or `"No"`.
- `Adjudication`: total annotated questions.
- `Final_Gap_Code` / `Gap_normalization`: number of normalized gaps.

In [397]:
# --- executions ---
n_executions = len(executions)
valid_executions = int((executions["Status"] == "Valida").sum())
print(f"Registered executions: {n_executions}")
print(f"Valid executions (Status == 'Valida'): {valid_executions}")

# --- raw_info ---
n_raw = len(raw_info)
dup_flag = raw_info["Duplicate_In_Run"].astype(str)
duplicates_removed = int(dup_flag.str.startswith("Duplicate of").sum())
questions_kept = int(dup_flag.isin(["Primary", "No"]).sum())
print(f"\nExtracted rows (raw_info): {n_raw}")
print(f"Duplicates removed: {duplicates_removed}")
print(f"Questions kept for annotation: {questions_kept}")

# --- Adjudication ---
annotated_questions = len(adj)
print(f"\nAnnotated questions (Adjudication): {annotated_questions}")

# --- normalized gaps ---
final_gap_count = int(adj["Final_Gap_Code"].nunique())
norm_gap_count = int(gap_norm["Gap_Code"].nunique())
print(f"Distinct Final_Gap_Code: {final_gap_count}")
print(f"Codes in Gap_normalization: {norm_gap_count}")

extraction_summary = pd.DataFrame([
    {"metric": "registered_executions", "value": n_executions},
    {"metric": "valid_executions", "value": valid_executions},
    {"metric": "extracted_question_lines", "value": n_raw},
    {"metric": "internal_duplicates_removed", "value": duplicates_removed},
    {"metric": "annotated_questions", "value": annotated_questions},
    {"metric": "normalized_gaps", "value": final_gap_count},
])
extraction_summary.to_csv(RESULTS_DIR / "extraction_summary.csv", index=False)
print(f"Exported: {(RESULTS_DIR / 'extraction_summary.csv').relative_to(REPO_ROOT)}")
extraction_summary

Registered executions: 120
Valid executions (Status == 'Valida'): 120

Extracted rows (raw_info): 1039
Duplicates removed: 441
Questions kept for annotation: 598

Annotated questions (Adjudication): 598
Distinct Final_Gap_Code: 64
Codes in Gap_normalization: 64
Exported: results\extraction_summary.csv


,metric,value
0,registered_executions,120
1,valid_executions,120
2,extracted_question_lines,1039
3,internal_duplicates_removed,441
4,annotated_questions,598
5,normalized_gaps,64


## 4. judge agreement

For each closed variable we compute the contingency matrix, the number of exact
agreements, the raw agreement, and Cohen's Kappa:

- `Category_J1` vs `Category_J2` (contextual category)
- `Coverage_J1` vs `Coverage_J2` (informational coverage)
- `Relevance_J1` vs `Relevance_J2` (functional relevance)

In [398]:
VARIABLES = [
    ("Category_J1", "Category_J2", "contextual category"),
    ("Coverage_J1", "Coverage_J2", "coverage"),
    ("Relevance_J1", "Relevance_J2", "functional relevance"),
]


def agreement_row(col_j1, col_j2, label):
    a = adj[col_j1].astype(str)
    b = adj[col_j2].astype(str)
    exact = int((a == b).sum())
    return {
        "variable": label,
        "n": len(a),
        "exact_agreements": exact,
        "raw_agreement": exact / len(a),
        "cohen_kappa": cohen_kappa_score(a, b),
    }


agreement_summary = pd.DataFrame([agreement_row(*v) for v in VARIABLES])
agreement_summary.to_csv(RESULTS_DIR / "judge_agreement_summary.csv", index=False)
print(f"Exported: {(RESULTS_DIR / 'judge_agreement_summary.csv').relative_to(REPO_ROOT)}")
agreement_summary


Exported: results\judge_agreement_summary.csv


,variable,n,exact_agreements,raw_agreement,cohen_kappa
0,contextual category,598,441,0.737458,0.443731
1,coverage,598,426,0.712375,0.411296
2,functional relevance,598,426,0.712375,0.243884


In [399]:
pd.crosstab(
    adj["Category_J1"].astype(str),
    adj["Category_J2"].astype(str),
    rownames=["J1"],
    colnames=["J2"],
)

J2,Decisional,Lexical,Operational,Systemic
J1,,,,
Decisional,36,3,55,0
Lexical,0,38,31,0
Operational,1,4,356,0
Systemic,0,3,60,11


In [400]:
pd.crosstab(
    adj["Coverage_J1"].astype(str),
    adj["Coverage_J2"].astype(str),
    rownames=["J1"],
    colnames=["J2"],
)

J2,Partially answered,Redundant,Unanswered
J1,,,
Partially answered,105,4,128
Redundant,18,6,3
Unanswered,19,0,315


In [401]:
pd.crosstab(
    adj["Relevance_J1"].astype(str),
    adj["Relevance_J2"].astype(str),
    rownames=["J1"],
    colnames=["J2"],
)

J2,Necessary,Useful
J1,,
Necessary,376,14
Useful,158,50


In [402]:
triple_agreement = int((
    (adj["Category_J1"] == adj["Category_J2"]) &
    (adj["Coverage_J1"] == adj["Coverage_J2"]) &
    (adj["Relevance_J1"] == adj["Relevance_J2"])
).sum())
print(f"Rows with simultaneous agreement on all three closed variables: {triple_agreement}")

needs_counts = adj["Needs_Adjudication"].value_counts()
no_adjudication = int(needs_counts.get("No", 0))
adjudicated_rows = int(needs_counts.get("Yes", 0))
print(f"Needs_Adjudication == 'No' (unanimous, no adjudication): {no_adjudication}")
print(f"Needs_Adjudication == 'Yes' (adjudicated rows): {adjudicated_rows}")

Rows with simultaneous agreement on all three closed variables: 188
Needs_Adjudication == 'No' (unanimous, no adjudication): 121
Needs_Adjudication == 'Yes' (adjudicated rows): 477


## 5. Building the per-execution metrics

`metrics_by_run` is built from `Adjudication`, grouped by `Run_ID`, `US_ID`, and
`Condition`:
- `annotated_questions`: count of `Classification_ID`;
- `distinct_normalized_gaps`: distinct `Final_Gap_Code`;
- `necessary_unanswered`: rows with `Final_Coverage == "Unanswered"` and `Final_Relevance == "Necessary"`;
- `redundant_questions`: rows with `Final_Coverage == "Redundant"`.

`Loop` is extracted from the `Run_ID` suffix (e.g. `US02_C0_R1` -> `Loop = 1`).

In [403]:
def loop_from_run(run_id):
    # Run_ID pattern: US<NN>_<COND>_R<loop>; conditions never contain 'R'.
    return int(str(run_id).rsplit("_R", 1)[-1])


records = []
for (run_id, us_id, condition), g in adj.groupby(["Run_ID", "US_ID", "Condition"]):
    records.append({
        "Run_ID": run_id,
        "US_ID": us_id,
        "Condition": condition,
        "Loop": loop_from_run(run_id),
        "annotated_questions": len(g),
        "distinct_normalized_gaps": int(g["Final_Gap_Code"].nunique()),
        "necessary_unanswered": int((
            (g["Final_Coverage"] == "Unanswered") & (g["Final_Relevance"] == "Necessary")
        ).sum()),
        "redundant_questions": int((g["Final_Coverage"] == "Redundant").sum()),
    })

metrics_by_run = pd.DataFrame(records).sort_values(["US_ID", "Condition", "Loop"]).reset_index(drop=True)

metrics_path = RESULTS_DIR / "metrics_by_run.csv"
metrics_by_run.to_csv(metrics_path, index=False)
print(f"Exported: {metrics_path.relative_to(REPO_ROOT)}")
metrics_by_run.head(12)

Exported: results\metrics_by_run.csv


,Run_ID,US_ID,Condition,Loop,annotated_questions,distinct_normalized_gaps,necessary_unanswered,redundant_questions
0,US02_C0_R1,US02,C0,1,5,5,4,0
1,US02_C0_R2,US02,C0,2,5,5,4,0
2,US02_C0_R3,US02,C0,3,5,5,3,0
3,US02_C0_R4,US02,C0,4,5,5,4,0
4,US02_C0_R5,US02,C0,5,5,5,4,0
5,US02_CD_R1,US02,CD,1,5,5,4,0
6,US02_CD_R2,US02,CD,2,5,5,4,0
7,US02_CD_R3,US02,CD,3,5,5,3,0
8,US02_CD_R4,US02,CD,4,5,5,3,0
9,US02_CD_R5,US02,CD,5,5,5,4,0


## 6. Descriptive statistics by condition

In [404]:
def bootstrap_ci(values, n_boot=N_BOOT, seed=SEED, alpha=0.05):
    rng = np.random.default_rng(seed)
    values = np.asarray(values, dtype=float)
    n = len(values)
    boot_means = values[rng.integers(0, n, size=(n_boot, n))].mean(axis=1)
    lo, hi = np.percentile(boot_means, [100 * alpha / 2, 100 * (1 - alpha / 2)])
    return lo, hi


summary_rows = []
for condition in CONDITIONS:
    vals = metrics_by_run.loc[metrics_by_run["Condition"] == condition, "necessary_unanswered"]
    lo, hi = bootstrap_ci(vals)
    summary_rows.append({
        "Condition": condition,
        "n": int(vals.shape[0]),
        "mean": vals.mean(),
        "median": vals.median(),
        "std": vals.std(ddof=1),
        "min": int(vals.min()),
        "max": int(vals.max()),
        "ci95_low": lo,
        "ci95_high": hi,
    })

condition_summary = pd.DataFrame(summary_rows).set_index("Condition").loc[CONDITIONS]
summary_path = RESULTS_DIR / "condition_summary.csv"
condition_summary.to_csv(summary_path)
condition_summary.round(4)

,n,mean,median,std,min,max,ci95_low,ci95_high
Condition,,,,,,,,
C0,20,2.90,3.0,0.7881,2,4,2.55,3.2500
CL,20,2.10,2.0,1.4832,0,4,1.45,2.7500
CO,20,0.65,0.5,0.7452,0,2,0.35,0.9512
CD,20,1.75,1.5,1.2927,0,4,1.20,2.3000
CS,20,1.70,1.0,1.4546,0,4,1.10,2.3500
CT,20,0.65,0.0,1.0894,0,3,0.20,1.1500


## 7. Statistical tests

`necessary_unanswered` was reshaped into a wide table indexed by (`US_ID`, `Loop`) with
one column per condition (20 rows = 4 User Stories x 5 repetitions).

- **Friedman** test across the six conditions.
- **Kendall's W** = Friedman statistic / (n_blocks x (k_conditions - 1)).
- **Wilcoxon** paired tests for C0 vs each contextual condition (`zero_method="wilcox"`),
  with **Holm** correction across the five comparisons.
- **Rank-biserial correlation** as the paired effect size (computed on condition - C0,
  so a reduction relative to C0 is negative).

In [405]:
wide = metrics_by_run.pivot_table(
    index=["US_ID", "Loop"], columns="Condition", values="necessary_unanswered"
)[CONDITIONS]
print(f"wide shape: {wide.shape}")
wide

wide shape: (20, 6)


Condition    C0   CL   CO   CD   CS   CT
US_ID Loop                              
US02  1     4.0  4.0  1.0  4.0  3.0  3.0
      2     4.0  4.0  1.0  4.0  4.0  2.0
      3     3.0  4.0  2.0  3.0  4.0  2.0
      4     4.0  4.0  2.0  3.0  4.0  3.0
      5     4.0  3.0  1.0  4.0  4.0  2.0
US08  1     3.0  3.0  0.0  0.0  2.0  0.0
      2     3.0  2.0  0.0  0.0  1.0  0.0
      3     2.0  1.0  0.0  1.0  1.0  0.0
      4     3.0  2.0  0.0  0.0  2.0  0.0
      5     2.0  1.0  0.0  1.0  1.0  0.0
US19  1     2.0  2.0  0.0  1.0  1.0  0.0
      2     2.0  3.0  0.0  1.0  0.0  0.0
      3     2.0  2.0  0.0  2.0  1.0  0.0
      4     2.0  4.0  0.0  1.0  3.0  0.0
      5     2.0  2.0  0.0  1.0  1.0  1.0
US22  1     3.0  0.0  2.0  2.0  0.0  0.0
      2     3.0  0.0  1.0  2.0  1.0  0.0
      3     4.0  0.0  1.0  2.0  0.0  0.0
      4     3.0  0.0  1.0  1.0  1.0  0.0
      5     3.0  1.0  1.0  2.0  0.0  0.0

In [406]:
friedman_stat, friedman_p = stats.friedmanchisquare(*[wide[c] for c in CONDITIONS])

n_blocks, k_conditions = wide.shape
kendall_w = friedman_stat / (n_blocks * (k_conditions - 1))
print(f"Friedman chi2 = {friedman_stat:.4f}")
print(f"Friedman p-value = {friedman_p:.4e}")
print(f"Kendall's W = {kendall_w:.4f}")

Friedman chi2 = 57.6427
Friedman p-value = 3.7271e-11
Kendall's W = 0.5764


In [407]:
def holm_correction(pvalues):
    p = np.asarray(pvalues, dtype=float)
    m = len(p)
    order = np.argsort(p)
    adjusted = np.empty(m)
    running_max = 0.0
    for rank, idx in enumerate(order):
        val = min(1.0, (m - rank) * p[idx])
        running_max = max(running_max, val)
        adjusted[idx] = running_max
    return adjusted

def rank_biserial(x, y):
    d = np.asarray(x, dtype=float) - np.asarray(y, dtype=float)
    d = d[d != 0]
    ranks = stats.rankdata(np.abs(d))
    r_plus = ranks[d > 0].sum()
    r_minus = ranks[d < 0].sum()
    return (r_plus - r_minus) / (r_plus + r_minus)

In [408]:
def run_family(pairs, labels):
    """Wilcoxon de cada par (treatment, baseline), com Holm dentro da família.

    O tamanho de efeito (rank-biserial) é orientado como treatment - baseline,
    de modo que uma redução do treatment em relação ao baseline fica negativa.
    """
    rows, raw = [], []
    for (treatment, baseline), label in zip(pairs, labels):
        diff = wide[treatment] - wide[baseline]
        if (diff == 0).all():
            stat, p = 0.0, 1.0
        else:
            # mode="approx" (aproximação normal) é o correto na presença de empates
            # e independe do heurístico "auto" do scipy, que varia entre versões.
            w = stats.wilcoxon(wide[treatment], wide[baseline],
                               zero_method="wilcox", mode="approx")
            stat, p = w.statistic, w.pvalue
        raw.append(p)
        rows.append({
            "comparison": label,
            "wilcoxon_stat": stat,
            "p_raw": p,
            "rank_biserial": rank_biserial(wide[treatment], wide[baseline]),
        })
    for row, p_adj in zip(rows, holm_correction(raw)):
        row["p_holm"] = p_adj
    return rows


# RQ1: cada condição contextual comparada ao baseline sem contexto (C0).
rq1_rows = run_family([(cond, "C0") for cond in CONTEXTUAL],
                      [f"C0 vs {cond}" for cond in CONTEXTUAL])

# RQ2: o contexto combinado (CT) comparado a cada contexto isolado.
ISOLATED = ["CL", "CO", "CD", "CS"]
rq2_rows = run_family([("CT", cond) for cond in ISOLATED],
                      [f"CT vs {cond}" for cond in ISOLATED])

In [409]:
# RQ1 family + omnibus Friedman/Kendall -> statistical_tests.csv
statistical_tests = pd.DataFrame(rq1_rows)
statistical_tests.insert(0, "n_blocks", n_blocks)
statistical_tests["friedman_chi2"] = friedman_stat
statistical_tests["friedman_p"] = friedman_p
statistical_tests["kendall_w"] = kendall_w
statistical_tests.to_csv(RESULTS_DIR / "statistical_tests.csv", index=False)

# RQ2 family (CT vs isolated) -> rq2_ct_vs_isolated_tests.csv
rq2_tests = pd.DataFrame(rq2_rows)
rq2_tests.to_csv(RESULTS_DIR / "rq2_ct_vs_isolated_tests.csv", index=False)

print(f"Exported: {(RESULTS_DIR / 'statistical_tests.csv').relative_to(REPO_ROOT)}")
print(f"Exported: {(RESULTS_DIR / 'rq2_ct_vs_isolated_tests.csv').relative_to(REPO_ROOT)}")
print("\nRQ1 (C0 vs contextual):")
print(statistical_tests[["comparison", "p_holm", "rank_biserial"]].to_string(index=False))
print("\nRQ2 (CT vs isolated):")
rq2_tests

Exported: results\statistical_tests.csv
Exported: results\rq2_ct_vs_isolated_tests.csv

RQ1 (C0 vs contextual):
comparison   p_holm  rank_biserial
  C0 vs CL 0.038875      -0.637363
  C0 vs CO 0.000293      -1.000000
  C0 vs CD 0.001242      -1.000000
  C0 vs CS 0.002945      -0.856209
  C0 vs CT 0.000293      -1.000000

RQ2 (CT vs isolated):


,comparison,wilcoxon_stat,p_raw,rank_biserial,p_holm
0,CT vs CL,0.0,0.000355,-1.0,0.001420
1,CT vs CO,27.5,1.000000,0.0,1.000000
2,CT vs CD,0.0,0.000451,-1.0,0.001420
3,CT vs CS,0.0,0.000715,-1.0,0.001431


### Normality of the paired differences

Shapiro-Wilk test on the paired differences C0 - condition, for each contextual
condition.

In [410]:
normality_rows = []
for condition in CONTEXTUAL:
    diff = wide["C0"] - wide[condition]
    sh = stats.shapiro(diff)
    normality_rows.append({
        "comparison": f"C0 - {condition}",
        "shapiro_W": sh.statistic,
        "p_value": sh.pvalue,
        "normal_at_0.05": bool(sh.pvalue > 0.05),
    })

normality_tests = pd.DataFrame(normality_rows)
normality_path = RESULTS_DIR / "normality_tests.csv"
normality_tests.to_csv(normality_path, index=False)
print(f"Exported: {normality_path.relative_to(REPO_ROOT)}")
normality_tests

Exported: results\normality_tests.csv


,comparison,shapiro_W,p_value,normal_at_0.05
0,C0 - CL,0.927477,0.138072,True
1,C0 - CO,0.779507,0.000434,False
2,C0 - CD,0.822554,0.001912,False
3,C0 - CS,0.937990,0.219634,True
4,C0 - CT,0.873937,0.013788,False


## 8. Final distributions

From `Adjudication`, the distributions of `Final_Coverage`, `Final_Relevance`, and
`Final_Category`, plus a few crossed counts used in the paper.

In [411]:
coverage_dist = adj["Final_Coverage"].value_counts()
coverage_table = (
    coverage_dist.rename_axis("Final_Coverage").reset_index(name="count")
)
coverage_table

,Final_Coverage,count
0,Unanswered,315
1,Partially answered,252
2,Redundant,31


In [412]:
relevance_dist = adj["Final_Relevance"].value_counts()
relevance_table = (
    relevance_dist.rename_axis("Final_Relevance").reset_index(name="count")
)
relevance_table

,Final_Relevance,count
0,Necessary,387
1,Useful,211


In [413]:
category_dist = adj["Final_Category"].value_counts()
category_table = (
    category_dist.rename_axis("Final_Category").reset_index(name="count")
)
category_table

,Final_Category,count
0,Operational,367
1,Decisional,91
2,Systemic,72
3,Lexical,68


In [414]:
unanswered_necessary = int((
    (adj["Final_Coverage"] == "Unanswered") & (adj["Final_Relevance"] == "Necessary")
).sum())
ct_unanswered = int((
    (adj["Condition"] == "CT") & (adj["Final_Coverage"] == "Unanswered")
).sum())
ct_unanswered_necessary = int((
    (adj["Condition"] == "CT") &
    (adj["Final_Coverage"] == "Unanswered") &
    (adj["Final_Relevance"] == "Necessary")
).sum())

crossed_counts = pd.DataFrame(
    [
        ("Unanswered + Necessary", unanswered_necessary),
        ("CT + Unanswered", ct_unanswered),
        ("CT + Unanswered + Necessary", ct_unanswered_necessary),
    ],
    columns=["metric", "count"],
)
crossed_counts

,metric,count
0,Unanswered + Necessary,195
1,CT + Unanswered,15
2,CT + Unanswered + Necessary,13


In [415]:
dist_records = []
for value, count in coverage_dist.items():
    dist_records.append({"dimension": "Final_Coverage", "value": value, "count": int(count)})
for value, count in relevance_dist.items():
    dist_records.append({"dimension": "Final_Relevance", "value": value, "count": int(count)})
for value, count in category_dist.items():
    dist_records.append({"dimension": "Final_Category", "value": value, "count": int(count)})
for value, count in [
    ("Unanswered+Necessary", unanswered_necessary),
    ("CT+Unanswered", ct_unanswered),
    ("CT+Unanswered+Necessary", ct_unanswered_necessary),
]:
    dist_records.append({"dimension": "crossed", "value": value, "count": int(count)})

final_distributions = pd.DataFrame(dist_records)
dist_path = RESULTS_DIR / "final_distributions.csv"
final_distributions.to_csv(dist_path, index=False)
print(f"Exported: {dist_path.relative_to(REPO_ROOT)}")
final_distributions

Exported: results\final_distributions.csv


,dimension,value,count
0,Final_Coverage,Unanswered,315
1,Final_Coverage,Partially answered,252
2,Final_Coverage,Redundant,31
3,Final_Relevance,Necessary,387
4,Final_Relevance,Useful,211
5,Final_Category,Operational,367
6,Final_Category,Decisional,91
7,Final_Category,Systemic,72
8,Final_Category,Lexical,68
9,crossed,Unanswered+Necessary,195


## 9. Elimination, persistence, emergence, and redundancy

For each contextual condition the set of `Final_Gap_Code` is compared and observed per
`US_ID` in C0 against the set observed in the target condition:

- **eliminated**: gap present in C0 for the US but absent in the condition;
- **emergent**: gap absent in C0 for the US but present in the condition;
- **persistent**: gap present in both, and its category matches the condition's target
  (CT targets all categories);
- **unchanged**: gap present in both, but outside the condition's target category;
- **redundant**: gap present in both, but every question for it in the condition was
  labeled `Redundant` (answer already available in the supplied material).

Each gap's canonical category is the mode of its `Final_Category`.

In [416]:
gap_category = adj.groupby("Final_Gap_Code")["Final_Category"].agg(lambda s: s.mode().iloc[0])
gaps_by_us = {
    cond: {us: set(g["Final_Gap_Code"].dropna()) for us, g in sub.groupby("US_ID")}
    for cond, sub in adj.groupby("Condition")
}

# Cobertura (Final_Coverage) de cada lacuna dentro de (condição, US).
coverage_by_gap = {
    (cond, us, gap): list(g["Final_Coverage"])
    for (cond, us, gap), g in adj.groupby(["Condition", "US_ID", "Final_Gap_Code"])
}


def is_redundant(condition, us, gap):
    """Lacuna reaparece na condição, mas todas as suas perguntas são redundantes,
    isto é, associadas a informação já disponível no material fornecido."""
    coverages = coverage_by_gap.get((condition, us, gap), [])
    return len(coverages) > 0 and all(c == "Redundant" for c in coverages)


EXPLANATION = {
    "eliminated": "present in C0, absent in condition",
    "emergent": "absent in C0, present in condition",
    "persistent": "present in both; category matches the condition target",
    "unchanged": "present in both; outside the condition target",
    "redundant": "present in both; all questions redundant in the condition",
}


def classify(condition, us, gap, in_c0, in_cond):
    if in_c0 and not in_cond:
        return "eliminated"
    if in_cond and not in_c0:
        return "emergent"
    # Presente em ambos: redundante tem prioridade sobre persistente/inalterado.
    if is_redundant(condition, us, gap):
        return "redundant"
    target = TARGET_CATEGORY[condition]
    return "persistent" if target is None or gap_category[gap] == target else "unchanged"


detail_records = []
for condition in CONTEXTUAL:
    c0_map, cond_map = gaps_by_us["C0"], gaps_by_us.get(condition, {})
    for us in sorted(set(c0_map) | set(cond_map)):
        c0_gaps, cond_gaps = c0_map.get(us, set()), cond_map.get(us, set())
        for gap in sorted(c0_gaps | cond_gaps):
            behavior = classify(condition, us, gap, gap in c0_gaps, gap in cond_gaps)
            detail_records.append((condition, us, gap, gap_category.get(gap),
                                   behavior, EXPLANATION[behavior]))

gap_behavior_details = pd.DataFrame(
    detail_records,
    columns=["Condition", "US_ID", "Gap_Code", "Final_Category", "Behavior", "Explanation"],
)
gap_behavior_summary = (
    gap_behavior_details.pivot_table(index="Condition", columns="Behavior",
                                     aggfunc="size", fill_value=0)
    .reindex(index=CONTEXTUAL,
             columns=["eliminated", "persistent", "emergent", "unchanged", "redundant"],
             fill_value=0)
)

gap_behavior_summary.to_csv(RESULTS_DIR / "gap_behavior_summary.csv")
gap_behavior_details.to_csv(RESULTS_DIR / "gap_behavior_details.csv", index=False)
gap_behavior_summary

Behavior,eliminated,persistent,emergent,unchanged,redundant
Condition,,,,,
CL,7,3,13,25,0
CO,9,14,10,10,2
CD,7,4,10,23,1
CS,7,5,14,21,2
CT,9,21,11,0,5


In [417]:
gap_behavior_details.head(15)

,Condition,US_ID,Gap_Code,Final_Category,Behavior,Explanation
0,CL,US02,G01,Systemic,unchanged,present in both; outside the condition target
1,CL,US02,G02,Decisional,unchanged,present in both; outside the condition target
2,CL,US02,G03,Operational,unchanged,present in both; outside the condition target
3,CL,US02,G04,Operational,unchanged,present in both; outside the condition target
4,CL,US02,G05,Lexical,eliminated,"present in C0, absent in condition"
5,CL,US02,G06,Systemic,unchanged,present in both; outside the condition target
6,CL,US02,G07,Decisional,eliminated,"present in C0, absent in condition"
7,CL,US02,G09,Systemic,emergent,"absent in C0, present in condition"
8,CL,US02,G11,Operational,emergent,"absent in C0, present in condition"
9,CL,US02,G12,Systemic,emergent,"absent in C0, present in condition"


## 10. Category-specific elimination

Breakdown of the eliminated gaps by their canonical category for each isolated
context condition (paper Table 6). Column totals match the `eliminated` column
of the behavior summary.

In [418]:
CATEGORY_ORDER = ["Lexical", "Operational", "Decisional", "Systemic"]
eliminated_gaps = gap_behavior_details[gap_behavior_details["Behavior"] == "eliminated"]
category_elimination = (
    eliminated_gaps.pivot_table(index="Final_Category", columns="Condition",
                                aggfunc="size", fill_value=0)
    .reindex(index=CATEGORY_ORDER, columns=ISOLATED, fill_value=0)
)
category_elimination.to_csv(RESULTS_DIR / "category_elimination_matrix.csv")
print(f"Exported: {(RESULTS_DIR / 'category_elimination_matrix.csv').relative_to(REPO_ROOT)}")
category_elimination

Exported: results\category_elimination_matrix.csv


Condition,CL,CO,CD,CS
Final_Category,,,,
Lexical,1,0,2,1
Operational,1,2,1,1
Decisional,2,2,2,3
Systemic,3,5,2,2
